# MECHA for developers, 0: overview of the package

**For:** developers who want to read, check, change or couple the model. **You need:** the user tutorial
`../user/01_first_run.ipynb` (what the model does), Python, numpy. **Run time:** a few seconds.

This series:

| | notebook | what it covers |
|---|---|---|
| 0 | `00_overview.ipynb` | the package's modules, the original MECHA and its reactions branch, how the port is validated, the docs |
| 1 | `01_granap_mecha.ipynb` | GRANAP's anatomies and MECHA's hydraulics coupled in a plant model; MECHA's inputs by role; scenarios; the barriers |
| 2 | `02_soil_gradient.ipynb` | the plant problem with a soil drier on one side of a root section |
| 3 | `03_needle_solutes.ipynb` | a pine needle of three sections, its transpiration, MechaSolutes over a day |
| 4 | `04_full_plant.ipynb` | a whole seedling in its soil, run to the fixed point |
| 5 | `05_extend_the_component.ipynb` | add a variable and a process to the component, run and test it |
| 6 | `06_solute_storage_and_decay.ipynb` | the solutes' storage fractions, their first-order loss and the liquid boundary |

## 1. Where the model comes from

`openalea.mecha` is **MECHA** (Couvreur et al., 2018) ported to metafspm components, on **GRANAP**'s anatomies.
Two versions of the original are kept, frozen, as the port's references:

- `openalea.mecha.legacy`, the refactored MECHA, which reads a cross section from an XML file (or a GRANAP network)
  and builds its own network: the hydraulics' reference;
- `openalea.mecha.legacy_reactions`, its *reactions* branch (solutes, osmotic terms, the needle): the solutes'
  reference.

The docs' **legacy** page (`doc/legacy.rst`) describes both, their algorithms, and every departure from them with
what the port does about it. The equations, each with the function computing it, are in `doc/equations.rst`, the
timings in `doc/performance.rst`.

## 2. The modules

Each module's first docstring line:

In [1]:
import importlib
import pkgutil

import openalea.mecha as package

for module in pkgutil.iter_modules(package.__path__):
    if module.name.startswith("legacy"):
        continue
    doc = (importlib.import_module(f"openalea.mecha.{module.name}").__doc__ or "").strip()
    print(f"{module.name:12} {doc.splitlines()[0] if doc else ''}")

component    MECHA as a metafspm functional component on the MPG DataStructure (doc/equations.rst, doc/legacy.rst).
conductance  MECHA's conductances (doc/equations.rst): the conductance K of every edge of every section, and the
geometry     MECHA's own geometry of a cross-section (doc/legacy.rst): the quantities the legacy derives in
osmotic      MECHA's osmotic presets (doc/equations.rst): the osmotic potentials of the cells and walls, and the reflection


plots        Plots of one section's hydraulics (the legacy's ``utils/visu`` views, as functions of arrays): its cells by tissue or
postprocess  MECHA's section-scale results from a solution (doc/equations.rst): the soil uptake and the radial
simulation   MECHA for users: the water flow across one root cross section in a few lines, its results as tables.
solutes      MECHA's solutes (doc/equations.rst): sucrose, mobile, and starch, immobile, on the liquid network of every


- `component.py` holds the components: `MechaSections` (the sections' geometry from GRANAP's variables) and
  `MechaHydraulics` (the conductances, MECHA's standard problem and the plant problem). Developer tutorial 1 couples
  them with GRANAP.
- `conductance.py` computes the walls', membranes' and plasmodesmata's conductances, the boundary terms and the
  vapour path; `geometry.py` builds a section's geometry from its arrays.
- `osmotic.py` (`MechaOsmoticPresets`) and `solutes.py` (`MechaSolutes`) give the osmotic potentials and the sugars'
  transport (developer tutorial 3).
- `postprocess.py` holds kr, the uptake and the axial conductances; `plots.py` the section figures.
- `simulation.py` is the users' entry point (`simulate`, `simulate_root`).

## 3. How the port is validated

The tests, in `test/`, run with `pytest test --ignore=test/legacy` from the repository's root (`test/legacy` holds the
original's own tests). They are in two folders:

In [2]:
import ast
from pathlib import Path

for folder in ("component", "comparison"):
    print(f"test/{folder}/")
    for path in sorted(Path("../../../test", folder).glob("test_*.py")):
        doc = ast.get_docstring(ast.parse(path.read_text())) or ""
        print(f"  {path.name:34} {' '.join(doc.split())[:110]}")

test/component/
  test_conductance_component.py      MECHA's conductances on GRANAP's anatomies: per edge and boundary node, by segment barrier and plant parameter
  test_geometry_component.py         MECHA's component on GRANAP's anatomies: its geometry written on the DataStructure, once per anatomy.
  test_legacy_fixes.py               The legacy's problems fixed in the port (devplan_metafspm.md, §18): each fix's rule, where the legacy differs.
  test_osmotic_component.py          The osmotic potentials (devplan_metafspm.md, §20): MECHA's presets as the "Solutes" inputs, the membranes' osm
  test_plant_problem.py              Step 5 (devplan_metafspm.md, §12–13): the axial junctions' conductances, and the plant problem, the sections j
  test_shoot.py                      The shoot in MECHA's plant problem (devplan_metafspm.md, §15, E1–E3): a stem element bearing a leaf element an
  test_simulation.py                 The users' entry point (openalea.mecha.simulation): its summary agre

- `test/comparison/` runs the same sections through the original and the port and compares every conductance, the
  matrix, the potentials, kr and the solutes, with the port's options set to reproduce the original
  (`test/comparison/legacy_run.py`: its plasmodesmata rules, its one wall thickness...).
- `test/component/` tests the port's own behaviour, each fix of the original by the rule it restores
  (`test_legacy_fixes.py`), and the users' entry point.
- `example/comparison/` draws the same comparisons as figures.

To rerun one file while working: `pytest test/component/test_simulation.py -q`.

## 4. What to read next

- `01_granap_mecha.ipynb` couples GRANAP's anatomies and MECHA's hydraulics in a plant model; 02–04 extend it to a
  soil gradient, a needle's solutes and a whole plant in its soil.
- `05_extend_the_component.ipynb` adds a variable and a process.
- GRANAP's tutorials explain the anatomies. metafspm's tutorial
  (`examples/soil_plant_atmosphere/tutorial_fspm2026_general_usage.ipynb` in the metafspm repository) builds
  components and couples them step by step; its user guide, `docs/user.md`, describes every part of the
  framework.